# Stocks Analysis

In [3]:
import tkinter as tk
from tkinter import filedialog

def select_excel_file():
    """
    Opens a file dialog to select an Excel file and returns its path.
    
    Returns:
        str or None: The full path to the selected Excel file, or None if no file was selected.
    """
    root = tk.Tk()
    root.withdraw()  # Hide the main window

    file_path = filedialog.askopenfilename(
        title="Select an Excel file",
        filetypes=[("Excel files", "*.xlsx *.xls *.xlsm"), ("All files", "*.*")]
    )
    
    root.destroy()  # Clean up the Tkinter root window
    return file_path if file_path else None

In [ ]:
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import sys

# Then use yf as normal
data = yf.download("LEVI")

def main(excel_file, num_years, top_n, investment_amount):
    try:
        df = pd.read_excel(excel_file, sheet_name="TICKERS", skiprows=2)
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        sys.exit(1)
    
    if 'Ticker' not in df.columns:
        print("Excel file must contain a column named 'Ticker'")
        sys.exit(1)
    
    tickers = df['Ticker'].dropna().astype(str).tolist()
    if not tickers:
        print("No tickers found in the Excel file.")
        sys.exit(1)
    
    end_date = datetime.today()
    start_date = end_date - timedelta(days=num_years * 365)
    one_year_ago = end_date - timedelta(days=365)
    
    results = []
    
    for ticker in tickers:
        try:
            # Download historical price data
            hist = yf.download(ticker, start=start_date, end=end_date, progress=False)
            if hist.empty:
                print(f"Warning: No price data for {ticker}. Skipping.")
                continue
            
            beginning_value = hist['Adj Close'].iloc[0]
            ending_value = hist['Adj Close'].iloc[-1]
            if beginning_value <= 0:
                print(f"Warning: Invalid beginning price for {ticker}. Skipping.")
                continue
            
            cagr = (ending_value / beginning_value) ** (1 / num_years) - 1

            # Fetch dividend data (last 1 year)
            ticker_obj = yf.Ticker(ticker)
            dividends = ticker_obj.dividends
            if not dividends.empty:
                # Filter dividends from the last 12 months
                recent_dividends = dividends[dividends.index >= one_year_ago]
                total_dividends = recent_dividends.sum()
            else:
                total_dividends = 0.0

            # Get current dividend yield (as a decimal, e.g., 0.02 = 2%)
            info = ticker_obj.info
            dividend_yield = info.get('dividendYield', None)
            if dividend_yield is None:
                dividend_yield_pct = "N/A"
            else:
                dividend_yield_pct = f"{dividend_yield:.2%}"

            current_price = hist['Adj Close'].iloc[-1]

            results.append({
                'Ticker': ticker,
                'CAGR': cagr,
                'Total_Dividends_Last_Year': total_dividends,
                'Dividend_Yield': dividend_yield_pct,
                'Current_Price': current_price
            })

        except Exception as e:
            print(f"Warning: Error processing {ticker}: {e}. Skipping.")
            continue
    
    if not results:
        print("No valid data to analyze.")
        sys.exit(1)
    
    # Sort by CAGR (descending)
    results.sort(key=lambda x: x['CAGR'], reverse=True)
    top_results = results[:top_n]
    
    investment_per_ticker = investment_amount / len(top_results)
    
    print(f"\nTop {len(top_results)} investment options based on {num_years}-year CAGR:")
    print("-" * 90)
    print(f"{'Ticker':<10} {'CAGR':<10} {'Div Yield':<12} {'Div Amount (1yr)':<18} {'Allocation':<12}")
    print("-" * 90)
    for r in top_results:
        print(
            f"{r['Ticker']:<10} "
            f"{r['CAGR']:>7.2%} "
            f"{r['Dividend_Yield']:<12} "
            f"${r['Total_Dividends_Last_Year']:<17.2f} "
            f"${investment_per_ticker:<11,.2f}"
        )

if __name__ == "__main__":
    excel_file = select_excel_file()
    #excel_file = "stocks_etfs.xlsx"
    num_years = 5
    top_n = 5
    investment_amount = 10000
    
    main(excel_file, num_years, top_n, investment_amount)

In [ ]:
pip install --upgrade certifi

In [ ]:
pip install --upgrade yfinance